# 04. Pelatihan dan Pemilihan Model

Membandingkan 3 algoritma (Logistic Regression, Decision Tree, Random Forest) via 5-Fold Stratified CV.

**Prasyarat:** Tahap 01 selesai.

**Keluaran:** Tiga model terlatih dan `training_notebook_state.json`.

Pilih kernel Python 3.12 dengan dependency proyek, lalu **Restart Kernel → Run All**. Tahap 01–05 dijalankan secara berurutan. Proyek edukasi data science, bukan diagnosis medis klinis.


## A. Inisialisasi Lingkungan dan Lokasi Proyek
Mempersiapkan konstanta fitur dan folder kerja.


In [1]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
from datetime import datetime, timezone
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'Dataset/Data_Asli/nhanes_depression_raw.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Buka notebook dari dalam folder proyek lengkap dengan Dataset.')
TARGET = "Depression"
LABELS = ["Not Depressed", "Depressed"]
DISPLAY = {
    "Not Depressed": "Tidak Terindikasi Depresi Mayor",
    "Depressed": "Berisiko Mengalami Depresi (PHQ-9 ≥ 10)"
}

CATEGORICAL = [
    "Gender", "Education Level", "Marital Status", "Household Income",
    "Trouble Sleeping History", "Vigorous Recreation", "Moderate Recreation",
    "Current Smoker", "Memory Problems", "Cant Work", "Limited Work"
]
NUMERIC = [
    "Age", "Sleep Hours", "Sedentary Time", "BMI", "Systolic", "Diastolic", "Pulse"
]
FEATURES = CATEGORICAL + NUMERIC

RANGES = {
    "Age": (18, 120),
    "Sleep Hours": (1.0, 24.0),
    "Sedentary Time": (0.0, 1440.0),
    "BMI": (10.0, 90.0),
    "Systolic": (60.0, 260.0),
    "Diastolic": (30.0, 160.0),
    "Pulse": (30.0, 220.0),
}

SEED = 42
REPORTS = ROOT / "Source_Code/reports"
EDA = REPORTS / "eda"
MODELING = REPORTS / "modeling"
ANALYSIS = REPORTS / "analysis"
PROCESSED = ROOT / "Dataset/Data_Olahan"
MODELS = ROOT / "Model"


for folder in [EDA, MODELING, ANALYSIS, PROCESSED, MODELS]:
    folder.mkdir(parents=True, exist_ok=True)
source = ROOT / 'Dataset/Data_Asli/nhanes_depression_raw.csv'
def save_json(path, value):
    path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False,
                   default=lambda x: x.item() if isinstance(x, np.generic) else str(x)),
        encoding="utf-8"
    )

def save_figure(path):
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches='tight')
    display(Image(filename=str(path)))
    plt.close()

print('Proyek:', ROOT)


Proyek: D:\KULIAH\sem-5\MACHINE LEARNING\PJBL\prediksi_gangguan_depresi


## B. Memuat Data dan Menyusun Fold CV
Hanya data latih yang digunakan untuk tuning.


In [2]:
split_file = PROCESSED / 'notebook_split.json'
if not split_file.exists():
    raise FileNotFoundError('Jalankan 01_persiapan_data.ipynb terlebih dahulu.')
split_state = json.loads(split_file.read_text(encoding='utf-8'))
for relative, expected in split_state['input_hashes'].items():
    if hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() != expected:
        raise ValueError('Data berubah. Jalankan ulang notebook 01: ' + relative)
raw = pd.read_csv(source, low_memory=False)
saved_clean = pd.read_csv(PROCESSED / 'cleaned.csv', low_memory=False)
clean = saved_clean[FEATURES + [TARGET]].copy()
ids = saved_clean['source_seqn']
profile_hashes = saved_clean['profile_hash'] if 'profile_hash' in saved_clean.columns else pd.util.hash_pandas_object(clean[FEATURES], index=False).astype(str)
conflict_hashes = set()
duplicate_mask = pd.Series([False] * len(raw))
X, y = clean[FEATURES], clean[TARGET]
train_idx, test_idx = split_state['train_indices'], split_state['test_indices']
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print('Data Latih:', X_train.shape, '| Data Uji:', X_test.shape)

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1)


Data Latih: (29007, 18) | Data Uji: (7252, 18)


## C. Definisi Pipeline Model
Setiap model digabungkan dengan pipeline preprocessing.


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

def make_preprocessor():
    """Membangun objek ColumnTransformer baru yang belum di-fit."""
    num_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ])
    cat_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    return ColumnTransformer([
        ("numeric", num_pipeline, NUMERIC),
        ("categorical", cat_pipeline, CATEGORICAL)
    ])

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


## D. Hyperparameter Tuning via GridSearchCV
Tuning menggunakan metrik evaluasi Macro-F1.


In [4]:
definitions = {
    "logistic_regression": (
        LogisticRegression(max_iter=500, random_state=SEED),
        {"model__C": [0.1, 1.0, 10.0], "model__class_weight": ["balanced", None]}
    ),
    "decision_tree": (
        DecisionTreeClassifier(random_state=SEED),
        {"model__max_depth": [4, 6, 8], "model__class_weight": ["balanced", None]}
    ),
    "random_forest": (
        RandomForestClassifier(random_state=SEED, n_jobs=-1),
        {"model__n_estimators": [100], "model__max_depth": [6, 10], "model__class_weight": ["balanced"]}
    )
}

fitted, rows = {}, []
for name, (model, params) in definitions.items():
    pipeline = Pipeline([("preprocess", make_preprocessor()), ("model", model)])
    baseline = cross_validate(pipeline, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)
    search = GridSearchCV(
        pipeline, params, cv=cv, scoring="f1_macro", n_jobs=-1,
        return_train_score=True, error_score="raise"
    )
    search.fit(X_train, y_train)
    fitted[name] = search.best_estimator_
    table = pd.DataFrame(search.cv_results_)
    table.to_csv(MODELING / f"cv_results_{name}.csv", index=False)
    best_idx = search.best_index_
    rows.append({
        "model": name,
        "baseline_cv_f1": float(np.mean(baseline["test_score"])),
        "cv_f1_mean": float(search.best_score_),
        "cv_f1_std": float(table.loc[best_idx, "std_test_score"]),
        "cv_train_f1": float(table.loc[best_idx, "mean_train_score"]),
        "best_params": json.dumps(search.best_params_)
    })
    joblib.dump(search.best_estimator_, MODELS / f"{name}_pipeline.joblib")
    print(f"{name} -> CV Macro-F1: {search.best_score_:.4f}")
display(pd.DataFrame(rows))


logistic_regression -> CV Macro-F1: 0.6216


decision_tree -> CV Macro-F1: 0.6299


random_forest -> CV Macro-F1: 0.6428


,model,baseline_cv_f1,cv_f1_mean,cv_f1_std,cv_train_f1,best_params
0,logistic_regression,0.621636,0.621636,0.003684,0.622393,"{""model__C"": 1.0, ""model__class_weight"": null}"
1,decision_tree,0.587510,0.629851,0.013954,0.651455,"{""model__class_weight"": null, ""model__max_dept..."
2,random_forest,0.603358,0.642809,0.007402,0.718279,"{""model__class_weight"": ""balanced"", ""model__ma..."


## E. Mengunci Pilihan Model dari CV Latih
Model terpilih dikunci sebelum melihat performa data uji.


In [5]:
selection = max(rows, key=lambda r: r['cv_f1_mean'])['model']
training_state = {
    'rows': rows, 'selection': selection, 'input_hashes': split_state['input_hashes'],
    'pipeline_hashes': {name: hashlib.sha256((MODELS / f'{name}_pipeline.joblib').read_bytes()).hexdigest() for name in fitted},
}
save_json(MODELING / 'training_notebook_state.json', training_state)
print('Model Terpilih CV:', selection)
print('Tahap 04 Selesai. Lanjut ke 05_evaluasi_dan_ekspor.ipynb')


Model Terpilih CV: random_forest
Tahap 04 Selesai. Lanjut ke 05_evaluasi_dan_ekspor.ipynb
